In [6]:
import json
import chromadb
from chromadb.utils import embedding_functions
import pandas as pd
import pytrec_eval
from tqdm import tqdm
from time import time
import torch


In [7]:
##load propositions
path = '/scratch/NLU/cvlachos/SCQA/Samu_XLSR_finetuning/data/propositions.json'

with open(path) as f:
    d = json.load(f)


propositions = d[0]['Propositions']
prop_dict = {i:f'id_{n}' for n, i in enumerate(propositions)}


In [ ]:
##load embedding model and Vector DB
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/LaBSE"
)

client = chromadb.PersistentClient(path="/scratch/NLU/cvlachos/SCQA/Samu_XLSR_finetuning/data/chroma_db")
collection = client.get_or_create_collection(
    name="propositions_VS",
    embedding_function=sentence_transformer_ef
)

In [8]:
##load dataframe
dataframe_path = '/scratch/NLU/cvlachos/SCQA/Samu_XLSR_finetuning/data/step4_fixed_props_asr.parquet'
df = pd.read_parquet(dataframe_path)[:]
df['propositions_ids'] = df.propositions.apply(lambda x : [prop_dict[i] for i in x])
df.head()

,context_qu_txt,rewrite,context_qu_audio,response,propositions,dialog_id,context_qu_asr,rewrite_asr,propositions_ids
0,"[Hello, Hello! How can I assist you today?, Wh...",What are supplementary policy benefits and are...,"[0_user.wav, 0_system.wav, 1_user.wav]",Supplementary policy benefits like Accidental ...,[Supplementary policy benefits like Accidental...,0,"[Hello., Hello. How can I assist you today?, W...",What are some of the monetary policy benefits ...,[id_3330]
1,"[Hello, Hello! How can I assist you today?, Wh...",How can a veteran get help with their insuranc...,"[0_user.wav, 0_system.wav, 1_user.wav, 1_syste...","If a veteran needs help with their claim, they...","[If a veteran needs help with their claim, the...",0,"[Hello., Hello. How can I assist you today?, W...",How can a veteran get help with their insuranc...,[id_3331]
2,"[Hello, Hello! How can I assist you today?, Wh...",What should a family member do if a service me...,"[0_user.wav, 0_system.wav, 1_user.wav, 1_syste...","For a service member who is terminally ill, a ...","[For a service member who is terminally ill, a...",0,"[Hello., Hello. How can I assist you today?, W...",What should a founding member do if a service ...,[id_3332]
3,"[Hello, Hello! How can I assist you today?, Wh...",How can a family member receive an insurance p...,"[0_user.wav, 0_system.wav, 1_user.wav, 1_syste...",For a family member to receive an insurance pa...,[For a family member to receive an insurance p...,0,"[Hello., Hello. How can I assist you today?, W...",How could a family member receive an insurance...,[id_3333]
4,"[Hello, Hello! How can I assist you today?, Wh...",Can veterans get free VA health care for servi...,"[0_user.wav, 0_system.wav, 1_user.wav, 1_syste...","Yes, veterans can get free VA health care for ...",[Veterans can get free VA health care for any ...,0,"[Hello., Hello. How can I assist you today?, W...",veterans get free VA health care for service c...,"[id_3334, id_3335]"


In [ ]:
eval_dict = {
    'map': 0.,
    'recip_rank': 0.,
    'recall_5': 0.,
    'recall_10': 0.,
    'recall_20': 0.,
    'recall_50': 0.,
    'recall_100': 0.,
  }

for row in tqdm(range(0,df.shape[0])):
    #print(df.iloc[row].rewrite)
    #query_embedding = sentence_transformer_ef([df.iloc[row].rewrite])
    #query_embedding = sentence_transformer_ef([df.iloc[row].context_qu_txt[-1]])

    
    query_embedding = sentence_transformer_ef([' [SEP] '.join(df.iloc[row].context_qu_txt[-3:])])

    ground_truth_props={'q':{k:1 for k in df.iloc[row].propositions_ids}}
    #print(ground_truth_props)

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=100
    )
    predictions = {'q':{k:1/(n+1) for n,k in enumerate(results['ids'][0])}}
    #print(predictions)

      
    evaluator = pytrec_eval.RelevanceEvaluator(
      ground_truth_props, {"map", "recip_rank", "recall_5", "recall_10", "recall_20", "recall_50", "recall_100"})

    results = evaluator.evaluate(predictions)
    #print(results)
    
    for i in results:
      for j in results[i]:
        eval_dict[j] += results[i][j]/df.shape[0]
    
    #break
    

100%|██████████| 5285/5285 [06:03<00:00, 14.55it/s]


All

In [33]:
eval_dict##rewrite

{'map': 0.3726610351739119,
 'recip_rank': 0.4534032496727469,
 'recall_5': 0.47082792483454455,
 'recall_10': 0.5528972913977722,
 'recall_20': 0.6284645460709996,
 'recall_50': 0.7272528643437223,
 'recall_100': 0.7986188104968115}

In [35]:
eval_dict##last turn 

{'map': 0.325448343833164,
 'recip_rank': 0.39713272964144786,
 'recall_5': 0.41716146147555716,
 'recall_10': 0.49625601508761097,
 'recall_20': 0.5635978974247748,
 'recall_50': 0.6614876708490955,
 'recall_100': 0.7349026176178821}

In [37]:
eval_dict##last turn + history

{'map': 0.10345088081164705,
 'recip_rank': 0.11902079883311867,
 'recall_5': 0.17957402998557412,
 'recall_10': 0.25719261554172185,
 'recall_20': 0.3419889681431789,
 'recall_50': 0.4556438116702975,
 'recall_100': 0.550379634031484}

Split 1000--1500

In [25]:
eval_dict##rewrite

{'map': 0.33106558312771767,
 'recip_rank': 0.4169450669530875,
 'recall_5': 0.43515158730158754,
 'recall_10': 0.5185611111111115,
 'recall_20': 0.6034230158730162,
 'recall_50': 0.6967246031746035,
 'recall_100': 0.7551626984126987}

In [23]:
eval_dict##last turn 

{'map': 0.29315059778405195,
 'recip_rank': 0.37082946763547514,
 'recall_5': 0.39028253968254006,
 'recall_10': 0.46699365079365107,
 'recall_20': 0.5488603174603178,
 'recall_50': 0.6296214285714288,
 'recall_100': 0.690784920634921}

In [21]:
eval_dict##last turn + history

{'map': 0.09473412886793854,
 'recip_rank': 0.1090122380438641,
 'recall_5': 0.1517214285714287,
 'recall_10': 0.2315357142857145,
 'recall_20': 0.3228198412698415,
 'recall_50': 0.4437912698412701,
 'recall_100': 0.5443015873015878}

Split 500--1000

In [14]:
eval_dict##rewrite

{'map': 0.33337175382159284,
 'recip_rank': 0.4166384221136684,
 'recall_5': 0.4315886446886451,
 'recall_10': 0.5204282051282054,
 'recall_20': 0.593252014652015,
 'recall_50': 0.6853876678876679,
 'recall_100': 0.7466800366300368}

In [16]:
eval_dict##last turn 

{'map': 0.26721704878407293,
 'recip_rank': 0.3370922969911038,
 'recall_5': 0.3495616605616608,
 'recall_10': 0.44375677655677676,
 'recall_20': 0.5113758241758243,
 'recall_50': 0.6011765567765568,
 'recall_100': 0.671049877899878}

In [18]:
eval_dict##last turn + history

{'map': 0.09813623215540453,
 'recip_rank': 0.12018851694192428,
 'recall_5': 0.1606857142857144,
 'recall_10': 0.23347142857142872,
 'recall_20': 0.30676825396825425,
 'recall_50': 0.4099967032967036,
 'recall_100': 0.4897474969474973}

Split 0--500

In [11]:
eval_dict##rewrite

{'map': 0.3763551708948753,
 'recip_rank': 0.4795868709254502,
 'recall_5': 0.4539067765567769,
 'recall_10': 0.5242091575091579,
 'recall_20': 0.6003996336996342,
 'recall_50': 0.7069020146520153,
 'recall_100': 0.7662520146520153}

In [9]:
eval_dict##last turn 

{'map': 0.32304450672957247,
 'recip_rank': 0.4070007751238305,
 'recall_5': 0.38611575091575123,
 'recall_10': 0.45713864468864507,
 'recall_20': 0.5199996336996341,
 'recall_50': 0.6229186813186818,
 'recall_100': 0.6887520146520151}

In [7]:
eval_dict##last turn + history

{'map': 0.1158469393392885,
 'recip_rank': 0.139586891796244,
 'recall_5': 0.18038956043956064,
 'recall_10': 0.25564340659340684,
 'recall_20': 0.3538010989010992,
 'recall_50': 0.46701776556776586,
 'recall_100': 0.5487391941391948}

Split 4500 - -1

In [10]:
eval_dict##last turn + history

{'map': 0.09902282394991727,
 'recip_rank': 0.1151968315269545,
 'recall_5': 0.1553216374269006,
 'recall_10': 0.23970760233918154,
 'recall_20': 0.31011695906432807,
 'recall_50': 0.424980506822613,
 'recall_100': 0.48192007797271064}

In [ ]:
eval_dict##last turn 

{'map': 0.3605547051348148,
 'recip_rank': 0.4339373956163092,
 'recall_5': 0.46312169312169416,
 'recall_10': 0.5354413812308559,
 'recall_20': 0.5842912837649692,
 'recall_50': 0.6802868281815665,
 'recall_100': 0.7358758006126445}

In [8]:
eval_dict##rewrite

{'map': 0.3905391983437215,
 'recip_rank': 0.4641084058423963,
 'recall_5': 0.4964550264550276,
 'recall_10': 0.5658507379560022,
 'recall_20': 0.6255193539404078,
 'recall_50': 0.7142049568365373,
 'recall_100': 0.7733027011974398}